# Flow drift metrics demo (A, P, F)

This notebook shows how to:
1) load logged `A/F/P` from disk to a chosen JAX device,
2) compute the drift metrics (fast/slow + dominant direction),
3) optionally render a quick video preview.

Supported log formats:
- **NPZ chunks** produced by `scripts/simulate_save_apf.py` (recommended)
- **Per-timestep pickles**: `{t}.pickle` or `{t}.zip` (gzip-pickled)


In [ ]:
import os
import sys

# Make repo root importable (so `import flow_drift_metrics` works)
sys.path.insert(0, os.getcwd())

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from flow_drift_metrics import (
    load_range,
    compute_drift_timeseries,
    iter_npz_snapshots,
)

print('JAX devices:', jax.devices())


## 0) Point to your logs

`save_pth` should be a directory containing either:
- chunked `P_steps_...npz` files (from `simulate_save_apf`), or
- per-timestep `{t}.pickle` / `{t}.zip` files.


In [ ]:
# Example path (edit this)
save_pth = "experiments/log_apf/checkpoints/2602021501"

# Step range (inclusive)
t1, t2 = 0, 20000

# Device selector for the loader/metric: "gpu", "gpu:0", "cpu", ...
has_gpu = any(d.platform == "gpu" for d in jax.devices())
device = "gpu:0" if has_gpu else "cpu"

print('save_pth:', save_pth)
print('range:', (t1, t2))
print('device:', device)


## 1) Load `A/F/P` to a chosen device (JAX tensors)

This is convenient for interactive analysis. For long sequences, prefer the streaming metric function below.


In [ ]:
batch = load_range(
    save_pth,
    t1=t1,
    t2=t2,
    fields=("A", "F", "P"),
    device=device,
    log_format="auto",  # or "npz" / "pickle"
)

t = batch["t"]
A = batch["A"]
F = batch["F"]
P = batch["P"]  # may be None for some pickle logs

print('t:', t.shape, t.dtype)
print('A:', A.shape, A.dtype, 'device:', A.device())
print('F:', F.shape, F.dtype, 'device:', F.device())
print('P:', None if P is None else (P.shape, P.dtype, 'device:', P.device()))


## 2) Compute drift metrics (streaming)

This reads frames from disk and runs a jitted metric per frame. You can optionally write a CSV.


In [ ]:
out = compute_drift_timeseries(
    save_pth,
    t1=t1,
    t2=t2,
    device=device,
    log_format="auto",  # or "npz" / "pickle"
    csv_path=None,       # e.g. "drift.csv"
    # You can override defaults here:
    r_pool=3,
    beta_t=0.01,
    q_top=0.995,
    m_thr_frac=0.02,
    mp_thr_frac=0.01,
    kappa_thr=0.2,
)

print(out.keys())
print('drift_fast:', out['drift_fast'].shape)
print('drift_slow:', out['drift_slow'].shape)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(out["t"], out["drift_fast"], label="drift_fast")
plt.plot(out["t"], out["drift_slow"], label="drift_slow")
plt.xlabel("t")
plt.ylabel("metric")
plt.legend()
plt.tight_layout()
plt.show()


## 3) Optional: render a video preview (Pcolor + |F|)

This uses the chunked-NPZ iterator and writes an MP4 with side-by-side frames.
Requires `imageio` + an ffmpeg backend.


In [ ]:
import imageio

def pcolor_rgb_u8(A_np: np.ndarray, P_np: np.ndarray) -> np.ndarray:
    A_np = A_np.astype(np.float32)
    P_np = P_np.astype(np.float32)
    inten = A_np.sum(axis=-1, keepdims=True)
    if P_np.shape[-1] >= 3:
        p3 = P_np[..., :3]
    else:
        reps = int(np.ceil(3 / P_np.shape[-1]))
        p3 = np.tile(P_np, (1, 1, reps))[..., :3]
    rgb = np.clip(inten * p3, 0.0, 1.0)
    return (rgb * 255).astype(np.uint8)

def flow_mag_u8(F_np: np.ndarray, percentile: float = 99.5, eps: float = 1e-8) -> np.ndarray:
    F_np = F_np.astype(np.float32)
    mag_ch = np.linalg.norm(F_np, axis=2)      # (H,W,C)
    mag = mag_ch.mean(axis=-1)                # (H,W)
    scale = np.percentile(mag, percentile)
    if not np.isfinite(scale) or scale < eps:
        scale = float(np.max(mag)) if float(np.max(mag)) >= eps else 1.0
    m = np.clip(mag / scale, 0.0, 1.0)
    m3 = (m[..., None] * 255).astype(np.uint8)
    return np.repeat(m3, 3, axis=2)

out_mp4 = "apf_preview.mp4"
fps = 30

writer = imageio.get_writer(out_mp4, fps=fps, codec="libx264")
try:
    for tt, sample in iter_npz_snapshots(save_pth, t1, t2, fields=("A", "P", "F")):
        rgb = pcolor_rgb_u8(sample["A"], sample["P"])
        flow = flow_mag_u8(sample["F"])
        frame = np.concatenate([rgb, flow], axis=1)
        writer.append_data(frame)
finally:
    writer.close()

print('Wrote', out_mp4)
